# Discriminant Classifiers

Using discriminant classifiers / Naive Bayes to classify our data.

In [1]:
import pandas as pd

In [ ]:
learn_data = pd.read_csv("../data/preprocess_train_v4.csv", header = None)
learn_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt',
       'LogSgot', 'Target']
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot,Target
0,48,0.511111,0.226640,4.615385,1.504077,5.641907,2.564949,4.304065,0
1,39,0.473684,-0.182383,3.115942,0.641854,5.192957,3.737670,4.127134,0
2,23,0.300000,-0.264120,3.100000,0.000000,5.356586,3.713572,4.382027,0
3,42,0.285714,-0.374875,3.018868,-0.356675,5.023881,3.555348,4.394449,0
4,54,0.504425,0.604032,4.250000,3.117950,6.324359,3.401197,3.610918,0


In [3]:
learn_data.isna().value_counts()

Age    DBRatio  ALBIScore  Glob   LogTB  LogAlkphos  LogSgpt  LogSgot  Target
False  False    False      False  False  False       False    False    False     450
Name: count, dtype: int64

In [4]:
X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

## Metrics

In [6]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

crossval_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])
validation_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear Discriminant Classifier

From PCA analysis, we know that our data can be separated in two clouds of sick and healthy patients respectively. A LD classifier might work well, but we know that the clouds may overlap. Also, their covariance matrices are clearly different. We might need to use a Quadratic Discriminant Classifier instead.

In [7]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.20, random_state = 42)

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
lda_model.fit(X_train, y_train)

print('Priors:', lda_model.priors_)

Priors: [0.5 0.5]


In [8]:
confusion(np.array(y_train), pd.Series(lda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	82	17
	0	111	150
Accuracy: 64.44%


In [9]:
confusion(np.array(y_val), pd.Series(lda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	24	5
	0	26	35
Accuracy: 65.56%


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

LDA_pipeline = Pipeline([('scaler', StandardScaler()), ('LDA', LinearDiscriminantAnalysis())])

n = 10
priors = [(p / n, 1 - p / n) for p in range(1, n)]

LDA_search = GridSearchCV(estimator = LDA_pipeline,
                          param_grid = {'LDA__priors' : priors},
                          scoring = 'f1_macro',
                          cv = 5)
LDA_search.fit(X_train, y_train)
LDA_search.best_params_

{'LDA__priors': (0.6, 0.4)}

In [15]:
LDA_search.best_score_

0.6446132973125455

In [14]:
from sklearn.model_selection import cross_validate

lda_priors = LDA_search.best_params_['LDA__priors']
lda_model = LinearDiscriminantAnalysis(priors = lda_priors)
lda_pipeline = Pipeline([('scaler', StandardScaler()), ('LDA', lda_model)])

cross_val_results = pd.DataFrame(cross_validate(lda_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["LDA", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.644613,0.666503,0.6415,0.688889


In [17]:
lda_pipeline.fit(X_train, y_train)
validation_df.loc["LDA", :] = compute_metrics(y_val, lda_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.62149,0.628321,0.619223,0.655556


## Quadratic Discrimant Classifier

We now use a QDA classifier. We see that the problem is preserved: the "sick" class overlaps too much with the healthy class and it

In [18]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
_ = qda_model.fit(X_train, y_train)

In [19]:
confusion(np.array(y_train), pd.Series(qda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	85	14
	0	120	141
Accuracy: 62.78%


In [20]:
confusion(np.array(y_val), pd.Series(qda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	27	2
	0	29	32
Accuracy: 65.56%


QDA can be regularized with a parameter between 0 and 1, so we can apply cross-validation in an attempt to obtain better metrics. In general, a small value of this regularization parameter (between 0.01 and 0.1) is desirable, but it does not improve results by much.

In [25]:
QDA_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', QuadraticDiscriminantAnalysis(priors = (0.5, 0.5)))])

n = 100
m = 10
regs = np.logspace(start = -4, stop = -0.5, num = n)
priors = [(p / m, 1 - p / m) for p in range(1, m)]

QDA_search = GridSearchCV(estimator = QDA_pipeline,
                          param_grid = {'QDA__reg_param' : regs,
                                        'QDA__priors' : priors},
                          scoring = 'f1_macro',
                          cv = 5)
QDA_search.fit(X_train, y_train)
QDA_search.best_params_

{'QDA__priors': (0.8, 0.19999999999999996),
 'QDA__reg_param': 0.31622776601683794}

In [26]:
QDA_search.best_score_

0.642161680613336

In [28]:
qda_priors = QDA_search.best_params_['QDA__priors']
qda_reg_param = QDA_search.best_params_['QDA__reg_param']
qda_model = QuadraticDiscriminantAnalysis(priors = qda_priors,
                                          reg_param = qda_reg_param)
qda_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', qda_model)])

cross_val_results = pd.DataFrame(cross_validate(qda_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["QDA", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.644613,0.666503,0.6415,0.688889
QDA,0.642162,0.658896,0.641275,0.691667


In [29]:
qda_pipeline.fit(X_train, y_train)
validation_df.loc["QDA", :] = compute_metrics(y_val, qda_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.65,0.652911,0.647895,0.688889
LDA,0.62149,0.628321,0.619223,0.655556


## Naive Bayes



In [30]:
from sklearn.naive_bayes import GaussianNB

gaussian_nb = GaussianNB(priors = (0.5, 0.5))
gaussian_nb.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(gaussian_nb.predict(X_train)))

		Predicted
		+1	0
Real	+1	86	13
	0	119	142
Accuracy: 63.33%


In [31]:
confusion(np.array(y_val), pd.Series(gaussian_nb.predict(X_val)))

		Predicted
		+1	0
Real	+1	26	3
	0	30	31
Accuracy: 63.33%


In [32]:
nb_pipeline = Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])

n = 10
priors = [(p / n, 1 - p / n) for p in range(1, n)]

nb_search = GridSearchCV(estimator = nb_pipeline,
                          param_grid = {'nb__priors' : priors},
                          scoring = 'f1_macro',
                          cv = 5)
nb_search.fit(X_train, y_train)
nb_search.best_params_

{'nb__priors': (0.9, 0.09999999999999998)}

In [34]:
nb_priors = nb_search.best_params_['nb__priors']
gaussian_nb = GaussianNB(priors = (0.5, 0.5))
nb_pipeline = Pipeline([('scaler', StandardScaler()), ('nb', gaussian_nb)])

cross_val_results = pd.DataFrame(cross_validate(nb_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["GaussianNB", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.644613,0.666503,0.6415,0.688889
QDA,0.642162,0.658896,0.641275,0.691667
GaussianNB,0.621222,0.701344,0.663923,0.630556


In [35]:
nb_pipeline.fit(X_train, y_train)
validation_df.loc["GaussianNB", :] = compute_metrics(y_val, nb_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.65,0.652911,0.647895,0.688889
GaussianNB,0.632198,0.702374,0.688025,0.633333
LDA,0.62149,0.628321,0.619223,0.655556


## Logistic Regression

In [36]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

logreg_model = LogisticRegression(C = 20, random_state = 42, class_weight = "balanced")

logreg_model.fit(X_train, y_train)
confusion(np.array(y_train), pd.Series(logreg_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	80	19
	0	102	159
Accuracy: 66.39%


/home/simple/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [37]:
confusion(np.array(y_val), pd.Series(logreg_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	21	8
	0	22	39
Accuracy: 66.67%


In [51]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression())])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs,
                                           'logreg__class_weight' : weights},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(X_train, y_train)
logreg_search.best_params_

{'logreg__C': 0.01747528400007685, 'logreg__class_weight': {0: 0.3, 1: 0.7}}

In [52]:
logreg_search.best_score_

0.6698802363872829

In [53]:
logreg_C = logreg_search.best_params_["logreg__C"]
logreg_weights = logreg_search.best_params_["logreg__class_weight"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = logreg_weights)
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["LogReg-Best", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
LDA,0.644613,0.666503,0.6415,0.688889
QDA,0.642162,0.658896,0.641275,0.691667
GaussianNB,0.621222,0.701344,0.663923,0.630556


In [54]:
logreg_pipeline.fit(X_train, y_train)
validation_df.loc["LogReg-Best", :] = compute_metrics(y_val, logreg_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
GaussianNB,0.632198,0.702374,0.688025,0.633333
LDA,0.62149,0.628321,0.619223,0.655556


## Trying our best classifiers on our test data

In [ ]:
test_data = pd.read_csv("../data/preprocess_test_v4.csv", header = None)
test_data.columns =  ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt', 'LogSgot']
test_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot
0,11,0.142857,-0.460075,3.000000,-0.356675,6.383507,3.258097,3.367296
1,62,0.500000,-0.172320,5.000000,0.587787,5.411646,4.234107,5.043425
2,60,0.285714,-0.460075,3.818182,-0.356675,5.159055,3.465736,2.639057
3,60,0.491228,0.226237,4.102564,1.740466,5.365976,6.021023,6.745236
4,48,0.222222,-0.260240,3.000000,-0.105361,5.164786,3.178054,3.988984


### QDA

In [ ]:
qda_pipeline.fit(X, y)

labels_qda = pd.DataFrame(columns = ['ID', 'Label'])
labels_qda['Label'] = pd.DataFrame(qda_pipeline.predict(test_data))
labels_qda['ID'] = labels_qda.index + 1
labels_qda.to_csv('../data/new_predictions/qda_best_fs.csv', index = False)

### Logistic Regression

In [ ]:
logreg_pipeline.fit(X, y)

labels_logreg = pd.DataFrame(columns = ['ID', 'Label'])
labels_logreg['Label'] = pd.DataFrame(logreg_pipeline.predict(test_data))
labels_logreg['ID'] = labels_logreg.index + 1
labels_logreg.to_csv('../data/new_predictions/logreg_best_fs.csv', index = False)